# Verified three-task portfolio: the public frontier of 2026-11-12, combined and re-verified

This version combines the best public file of each task as of 2026-11-12 10:00 UTC (version 5), with our invsqrt splice, and re-verifies all three with
the unchanged official verifiers. **Almost all of it is other people's work; every file and every improvement is
credited below, with the author, the change and a link.** Questions, reviews and corrections are welcome in the
comments.

| Task | File | Built by (latest change first) |
|---|---|---|
| invsqrt | `9c708f7b` (new) | **our splice** of WCao24's 44-CSWAP normaliser (`94b29cc5`) and Thomas Gamet's early prefix (`5ec4f516`), on maoxin123's and manwithacat's 62-qubit line |
| DQI | Atakan Aldemir's iso-gap h5 file | Atakan's four-Toffoli GF(64) inverse character, on our readout-window transplant (version 1 of this notebook) of Atakan's `bb4287e7` |
| chemistry | npz `e9cffb84` + 15 loaders | **maoxin123's v22 loaders, 4,089 Toffolis per pass** (new, -16 Toffolis per block encoding); Heliosli's update of prvsiyan's weights, on Lera (marculera)'s (8, 143, 44) table |

## Acknowledgements
Thank you to everyone whose public work this notebook stands on. In particular:
**Alex Chilton**, whose *Verified Three-Task Portfolio* set the standard for assembling, pinning and re-verifying the
public frontier, and whose credit tables we follow; **Atakan Aldemir** for the DQI file used here and the readout
windows, shared products and GF(64) characters behind it; **WCao24** for the invsqrt normaliser; **prvsiyan** for the
chemistry refit and the careful written explanations; **maoxin123** for the invsqrt base and the chemistry loaders
(the 4,089-Toffoli loaders used here); **Heliosli** for the weight update and the public *qcc-circuits-live* dataset;
**manwithacat (James Barlow)** for the HW-25 join and the public circuits dataset we used to find the readout windows;
**Ertuğrul Özer (ozertuu)** for the pinned cache and table; **Lera (marculera)** for the DF-THC table and words; **Kirill Tymchishin**, **Syed Asad Ali**,
**SomeJoint (Agniv Banerjee)**, **busyaprime (Akmal Xodarev)**, **yoyobar**, **mrmeows (team selenium)**, **Hosen42**,
**Gengsr**, **Ahmed Berat Özer**, **Lera (marculera)**, **Isabe11aM00re**, **Dmitrii Gluzdov**, **Leo Tang**,
**milos**, the team **i want to run gemma 9b**, **jday96314** (the decoder design) and **Oleg Sher (sheroleg)**, whose
explanatory notebook helped us read the invsqrt circuit. And Google Quantum AI and Kaggle for the competition, the
open kickmix simulator and the verifiers. If this notebook is useful to you, an upvote is appreciated. If we missed you or misattributed a change, please tell us in the comments
and we will fix it in the next version.

**What changed in version 5.** The chemistry is now maoxin123's new loaders (4,089 Toffolis per pass instead of 4,097)
with Heliosli's weight update, exactly as published in Heliosli's *qcc-circuits-live* dataset; invsqrt and DQI are
unchanged. All the credit for the chemistry gain goes to maoxin123 and Heliosli.

**What is ours in this version.** The invsqrt splice `9c708f7b`: 1,213.87 Toffolis per shot against 1,216.83 for
the best public circuit (official verifier, 93,555.26 vs 93,739.35, -0.20%), the combination and its verification, and the readout-window transplant inside
the DQI (version 1 of this notebook: four controlled-swap readout windows of the `e9f54be3` line moved onto
`bb4287e7`, -48 Toffolis per shot), which Atakan Aldemir's file builds on.
We also scanned 50 public DQI files region by region for further transplants: the only candidates (one Toffoli each)
depend on regions that this line has rewritten, and fail the official verifier when moved alone, so none is used.

In [ ]:
import sys, os, pathlib, zipfile, json, time, subprocess, hashlib, shutil, math, difflib, re
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
COMP = pathlib.Path("/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation")
KM = COMP / "kickmix"; VER = COMP / "verifiers"; sys.path.insert(0, str(KM))
import kickmix as km
inp = pathlib.Path("/kaggle/input"); SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
sha = lambda b: hashlib.sha256(b).hexdigest()
def members(zpath):
    with zipfile.ZipFile(zpath) as z: return {n: z.read(n) for n in z.namelist()}
zips = {str(p): p for p in inp.rglob("submission.zip")}
wcao = members(next(p for k, p in zips.items() if "deterministic-open-portfolio" in k))
atak = members(next(p for k, p in zips.items() if "iso-gap-dqi-h5" in k))
thom = members(next(p for k, p in zips.items() if "new-base-bundle" in k))
inv = wcao["invsqrt.kmb"]; assert sha(inv).startswith("94b29cc5"), sha(inv)
# chemistry: Heliosli's qcc-circuits-live dataset (maoxin123's 4,089-Toffoli loaders + Heliosli's weight update)
HEL = next(p.parent for p in inp.rglob("QCC_RELEASE_MANIFEST.json"))
man = json.load(open(HEL / "QCC_RELEASE_MANIFEST.json"))
chem = {f["file"]: (HEL / f["stored_as"]).read_bytes() for f in man["files"] if f["file"].startswith("dfthcsos/")}
for f in man["files"]:
    if f["file"] in chem: assert sha(chem[f["file"]]) == f["sha256"], f
npz = chem["dfthcsos/submission.npz"]; assert sha(npz).startswith("e9cffb84"), sha(npz)
assert sha(chem["dfthcsos/angle_k_0.kmb"]).startswith("77f5f6d8")
dqi = atak["dqi.kmb"]
log("WCao24 invsqrt", sha(inv)[:12], "Thomas invsqrt", sha(thom["invsqrt.kmb"])[:12], "dqi (Atakan h5)", sha(dqi)[:12], "chem npz", sha(npz)[:12])
(SUB / "invsqrt.kmb").write_bytes(inv); (SUB / "dfthcsos" / "submission.npz").write_bytes(npz)
for n, b in chem.items():
    if n.startswith("dfthcsos/angle_k_"): (SUB / n).write_bytes(b)
(SUB / "dqi.kmb").write_bytes(dqi)

## invsqrt: our splice of WCao24's normaliser and Thomas Gamet's early prefix
Two public improvements of the 62-qubit circuit sit in different places: WCao24's 44-CSWAP normaliser (`94b29cc5`)
and Thomas Gamet's early-prefix circuit (`5ec4f516`). We join public 62-qubit circuits at *clean cuts* (empty
condition stack, no classical bit written before the cut and read after it) where the wires and the phase agree on
every sampled input, and take the cheapest path (method and code in our analysis notebook). The segments below are
that path; the result is pinned by SHA-256 and verified by the official verifier further down.

In [ ]:
SEGMENTS = [["034c7c2a54d5", 0, 69], ["13b5bf7a18bc", 81, 83], ["26ebf39df3b1", 71, 92], ["94b29cc52689", 118, 479],
            ["034c7c2a54d5", 235, 421], ["5ec4f516164c", 515, 36611], ["034c7c2a54d5", 36245, 36409], ["5ec4f516164c", 36867, 36869],
            ["6c6a734dbcb6", 36857, 36859], ["94b29cc52689", 36549, 36553], ["13b5bf7a18bc", 53614, 53616], ["034c7c2a54d5", 36419, 36423],
            ["13b5bf7a18bc", 53620, 53622], ["26ebf39df3b1", 36634, 36636], ["ac540b8eff24", 46889, 46897], ["034c7c2a54d5", 36437, 36477]]
INV_SHA = "9c708f7b9abc7c7e6719f6e258a3881945e9ee1fd4fd7009fe37367ca1ef20d1"
D = pathlib.Path("/tmp/inv"); D.mkdir(exist_ok=True)
need = {s_[0] for s_ in SEGMENTS}; got = {}
for p in inp.rglob("*.kmb"):
    if "/invsqrt/" in str(p) and p.stem[:12] in need: got[p.stem[:12]] = p.read_bytes()
for b in (wcao["invsqrt.kmb"], thom["invsqrt.kmb"]): got[sha(b)[:12]] = b
for k, b in got.items():
    (D / f"{k}.kmb").write_bytes(b); km.Circuit.from_file(D / f"{k}.kmb", format="kmb").to_file(D / f"{k}.kmx", format="kmx")
def load_lines(name):
    head, body = [], []
    for line in open(D / f"{name}.kmx"):
        t = line.split()
        if not t: continue
        (head if t[0] in ("APPEND_TO_REGISTER", "REGISTER") else body).append(line.rstrip("\n"))
    return head, body
def remap_bits(line, table, counter):
    out = []
    for tok in line.split():
        if tok.startswith("b") and tok[1:].isdigit():
            if tok not in table: table[tok] = "b" + str(counter[0]); counter[0] += 1
            tok = table[tok]
        out.append(tok)
    return " ".join(out)

head, lines, counter = None, [], [0]
for name, a, b in SEGMENTS:
    h, body = load_lines(name); head = head or h; table = {}
    lines += [remap_bits(l, table, counter) for l in body[a:b]]
(D / "splice.kmx").write_text("\n".join(head + lines) + "\n")
km.Circuit.from_file(D / "splice.kmx", format="kmx").to_file(SUB / "invsqrt.kmb", format="kmb")
inv_sha = sha((SUB / "invsqrt.kmb").read_bytes()); log("invsqrt splice", inv_sha[:12])
assert inv_sha == INV_SHA, inv_sha

## Official verification of the exact bytes that go into `submission.zip`

In [ ]:
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"])
log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors", "repetition_factor")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
dqi_zero = res["dqi"]["score"] / res["dqi"]["repetition_factor"] * 1.1912604648115832
for name, dq in (("this run's DQI draw", res["dqi"]["score"]), ("DQI at zero failed shots", dqi_zero)):
    r = {"invsqrt": REF["invsqrt"] / res["invsqrt"]["score"], "dqi": REF["dqi"] / dq, "dfthcsos": REF["dfthcsos"] / res["dfthcsos"]["score"]}
    print(f"{name:26s}", {k: round(x, 6) for k, x in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 11, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res},
          open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written")

## Credits (who did what), with links
**invsqrt** `9c708f7b`
- asteriospanoras (this notebook): the splice of the two circuits below at clean cuts, -3.0 Toffolis vs the best one.
- Thomas Gamet (with OpenAI Codex): the early-prefix circuit `5ec4f516`
  ([Quantum Attempt - new base bundle](https://www.kaggle.com/code/thomasgamet/quantum-attempt-new-base-bundle)).
- WCao24: the 44-CSWAP input normaliser `94b29cc5`, -2.0 Toffolis
  ([VQCC Deterministic Open Portfolio](https://www.kaggle.com/code/wcao24/vqcc-deterministic-open-portfolio)).
- maoxin123: `a15b0d73`, Block A shift-and-add stages re-ordered, controlled-SWAP rewrites
  ([LB 25.303 | Combined invsqrt + refit chem](https://www.kaggle.com/code/maoxin123/lb-25-303-combined-invsqrt-refit-chem)).
- manwithacat (James Barlow): HW-25 join and the public circuits dataset
  ([VQCC Hedge Wizardry](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook)); Gengsr: the 46-CSWAP shifter
  ([LB 25.2268](https://www.kaggle.com/code/gengsr/lb-25-2268-shifter-46-own-chemistry-loaders)).
- The 62-qubit lineage: Syed Asad Ali (base circuit), Hosen42 (q25 CCX merge, measure-first rewrites), Kirill
  Tymchishin (carry port), Atakan Aldemir (CSWAP deletion), Ertuğrul Özer (`13b5bf7a` composition), busyaprime (Akmal
  Xodarev), yoyobar (19-bit conditional phase fusion), mrmeows / team selenium (the 63-qubit ancestor), Leo Tang and
  Ehan Mallick, as credited in their notebooks.

**DQI** (Atakan Aldemir's iso-gap h5 file)
- Atakan Aldemir: four-Toffoli GF(64) inverse character at 28 sites, -27.6 Toffolis per shot
  ([Quantum Circuits: Iso-Gap + DQI h5](https://www.kaggle.com/code/atakanaldemir/quantum-circuits-iso-gap-dqi-h5));
  earlier the five-Toffoli version and `bb4287e7`
  ([Shared Products and Controlled Refits](https://www.kaggle.com/code/atakanaldemir/qcc-shared-products-and-controlled-refits)),
  and the controlled-swap readout windows of `e9f54be3`
  ([Terminal Readout and Phase Folds](https://www.kaggle.com/code/atakanaldemir/qcc-terminal-readout-and-phase-folds)).
- asteriospanoras (this notebook, version 1): the readout windows transplanted onto `bb4287e7`, -48 Toffolis per shot.
- Team *i want to run gemma 9b*: `cd71c482` and `b4dc000c` (joint zero-chart rewrites, conditional reverse-phase
  schedule, prefix fusion), shipped publicly via Oleg Sher's notebook.
- Heliosli (`9fcda0c1`: retained products; the regions we transplanted were taken from this file), Kirill Tymchishin
  (measurement-order identities, retained-product audit), Syed Asad Ali (decoder `1ab3ac53`, measurement peephole),
  SomeJoint (narrow return), Ali Haydar Özdağ (flag-carry idea), Isabe11aM00re, MAIDANG, busyaprime and yoyobar (product
  caching), Leo Tang, mrmeows, manwithacat (release-and-recompute), daifanhao, Khattar and Gidney (2024, the paper the
  decoder follows), and jday96314 (the decoder design; its repository states no licence, so it is credited, not
  relicensed).

**Chemistry**
- maoxin123: v22 loaders, **4,089 Toffolis per pass** (five-block low-monomial subspace search), used from version 5
  ([QCC v22 | 4089-Toffoli chemistry loaders](https://www.kaggle.com/code/maoxin123/qcc-v22-4089-toffoli-chemistry-loaders)).
- Heliosli: weight update `e9cffb84` of prvsiyan's weights with prvsiyan's refit code, and the
  [qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live) dataset this version reads
  ([The dataset is public now](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)).
- Lera (marculera): the (8, 143, 44) DF-THC table and its angle words
  ([dfthc-c44-build](https://www.kaggle.com/datasets/marculera/dfthc-c44-build)), and the authors her NOTICE credits
  (Ahmed Berat Özer, kozykappa, Kozykorpesh and Sumit Chongder, mrmeows, michaelrustad, phucthaiv02, somejoint,
  gengsr, qwertykeypadapple, lightningv08, syed3000).
- prvsiyan: constrained refit of the weights `f21b8a87` (used up to version 4)
  ([Verifier-Guided Quantum Circuit Optimization](https://www.kaggle.com/code/prvsiyan/verifier-guided-quantum-circuit-optimization)).
- maoxin123: v22 loaders, 4,097 Toffolis per pass (used up to version 4)
  ([QCC v22 | 4097-Toffoli chemistry loaders](https://www.kaggle.com/code/maoxin123/qcc-v22-4097-toffoli-chemistry-loaders)).
- ozertuu (Ertuğrul Özer): table `8afd4c67` and the cache dataset. Syed Asad Ali: the 8 x 143 x 44 table and the
  loader compiler. Lera (marculera): the DF-THC pipeline and the C44 build. Ahmed Berat Özer: progressive rounding and
  group compression. Atakan Aldemir: the pi-gauge. Weight refits before prvsiyan's: Isabe11aM00re, maoxin123 (with
  manwithacat's `dfthc_refit`), Heliosli, busyaprime. Dmitrii Gluzdov, milos and Agniv Banerjee: earlier shapes.

**Assembly and verification practice**: Alex Chilton's [Verified Three-Task Portfolio](https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio)
(pinning by SHA-256, re-verifying with the stock verifiers, pricing a DQI file at its board draw), and WCao24's
deterministic, byte-reproducible build. **Analysis**: Oleg Sher's [Quantum Circuits from Zero](https://www.kaggle.com/code/sheroleg/quantum-circuits-from-zero-1-sqrt-x-explained).
Our companion analysis notebook: [QCC Toolkit: Simulator & Cost Anatomy](https://www.kaggle.com/code/asteriospanoras/qcc-28-06018-what-works-what-fails-and-why).
Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each file keeps its authors' terms; nothing here is
relicensed. The code of this notebook is Apache-2.0.